# Data collection and cleaning
Collects data from Arxiv, does basic data cleaning, and uploads it to supabase.

Requirements:
requests
feedparser
tqdm
supabase

In [7]:
from pathlib import Path
import sys

# annoying boilerplate that adds ../paper-clustering/ to the $PYTHONPATH variable 
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path)

from datetime import datetime, timedelta
import tarfile, io, os, re, time
import shutil
import tempfile
from tqdm import tqdm
from supabase import create_client, Client
import pandas as pd
from pprint import pp

from src.arxiv_fetch import download_from_arxiv

['/home/alfred/git/python/paper-clustering', '/home/alfred/miniconda3/envs/paper-clustering/lib/python314.zip', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.14', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.14/lib-dynload', '', '/home/alfred/miniconda3/envs/paper-clustering/lib/python3.14/site-packages']


In [8]:
def log_failure(entry, e, stage, output=None):
    """Logs when object `entry` causes an exception `e` at stage `stage` of the processing pipeline.
    Stores results in `output`, or prints to standard output if `output` is None.
    """
    entry_id = getattr(entry, "id", None)

    arxiv_id = None
    if isinstance(entry_id, str) and "/abs/" in entry_id:
        arxiv_id = entry_id.split("/abs/")[-1]

    failure = {
        "stage": stage,
        "failed_at": datetime.now().isoformat(),
        "error_type": type(e).__name__,
        "error_message": str(e),
        "title": getattr(entry, "title", None),
        "arxiv_id": arxiv_id,
        "entry_id": entry_id,
        "url": getattr(entry, "link", None),
    }

    if output is None:
        pp(failure)
    else:
        output.append(failure)

In [9]:
CATEGORIES = ["cs.DS", "cs.IT", "cs.CC", "math.CO"]
DAYS_BACK = 7
KEYWORDS = [
    "locally+decodable+code",
    "matrix+concentration",
    "coding+theory",
    "hypergraph",
    "random+tensor",
    "matching+vectors",
    "rainbow+cycle",
]
feed = download_from_arxiv(CATEGORIES, 0, KEYWORDS);
print(type(feed))

<class 'feedparser.util.FeedParserDict'>


In [3]:
def get_intro_text(session, arxiv_id):
    # --- download source ---
    url = f"https://arxiv.org/e-print/{arxiv_id}"

    try:
        r = session.get(url, timeout=60)
        if r.status_code != 200:
            return ""

        with tarfile.open(fileobj=io.BytesIO(r.content), mode="r:gz") as tar:
            for member in tar:
                if not member.isfile() or not member.name.endswith(".tex"):
                    continue

                f = tar.extractfile(member)
                if not f:
                    continue

                text = f.read().decode(errors="ignore")

                # Check for main document
                if "\\begin{document}" not in text:
                    continue

                # --- extract introduction ---
                m = re.search(
                    r"\\section\*?\{[^}]*[intro|Intro][^}]*\}(.*?)(?=\\section|\Z)",
                    text,
                    re.IGNORECASE | re.DOTALL
                )
                if not m:
                    return ""

                intro = m.group(1)

                # --- minimal LaTeX cleanup ---
                # Remove comments
                intro = re.sub(r"%.*", "", intro)

                # # Remove common commands (light cleanup)
                # intro = re.sub(r"\\[a-zA-Z]+\{.*?\}", "", intro)
                # intro = re.sub(r"\\[a-zA-Z]+", "", intro)
                # # 2. Replace all block math environments with a placeholder
                # intro = re.sub(r"\\[.*?\\]", " <MATH> ", intro, flags=re.DOTALL) # Display math (\[...\])
                # intro = re.sub(
                #     r"\\begin\{equation\}.*?\\end\{equation\}",
                #     " <MATH> ", intro, flags=re.DOTALL
                # )

                # # 3. Replace all inline math ($...$) with a placeholder
                # intro = re.sub(r"\$.*?\$", " <MATH> ", intro, flags=re.DOTALL)

                # # 4. Preserve content of commands with arguments (e.g., \textbf{content} -> content)
                # intro = re.sub(r"\\[a-zA-Z]+\*?\{([^}]*)\}", r"\1", intro)

                # # 5. Remove any remaining backslash-prefixed commands or artifacts (like \1, \item, \section etc.)
                # intro = re.sub(r"\\[^\\s]+", "", intro)

                # 6. Normalize whitespace
                intro = re.sub(r"\\s+", " ", intro)
                return intro.strip()
    except (tarfile.ReadError, OSError):
        return ""
    return ""